# DPO on Qwen2.5-0.5B-Instruct

This notebook fine-tunes Qwen2.5-0.5B-Instruct with Direct Preference Optimization
(DPO) on UltraFeedback preference pairs. It's small enough to run on a free Colab T4 GPU.

**Final test result on 200 unseen pairs**

| Metric | Original model | Final model |
|---|---|---|
| DPO loss | 0.693 | **0.606** |
| Moved toward the preferred answer | n/a | **68.5%** |

**Contents**

1. Setup
2. Load the data
3. Prepare the pairs
4. Understand the DPO loss
5. Save a baseline
6. Train on 2,000 pairs
7. Continue on new pairs
8. Review the results
9. Run the final test
10. Save the model

## 1. Setup

Use a T4 GPU runtime. The install cell pins the package versions used here. Its pip
warning about `cudf` and `pyarrow` is a conflict with a package this notebook doesn't use.

In [1]:
%pip install -q trl==1.17.1 transformers==5.22.1 datasets==5.1.0 accelerate==1.15.0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer

## 2. Load the data

Each UltraFeedback row holds one conversation with a preferred answer (`chosen`), one
with a less preferred answer (`rejected`), and a score for each. The prompt sits inside
those conversations as the first message, and the scores came from GPT-4.

In [3]:
data = load_dataset("trl-lib/ultrafeedback_binarized")

README.md:   0%|          | 0.00/643 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  131MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.14MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/62135 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [4]:
train = data['train']

### Load the tokenizer

The tokenizer turns each conversation into Qwen's chat format, then into token IDs.

In [5]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

## 3. Prepare the pairs

### Choose which pairs to keep

A pair is kept only if the chosen answer scored higher and both conversations fit in
512 tokens. Long pairs are dropped rather than cut, because a shortened answer may no
longer deserve its score.

In [6]:
MAX_TOKENS = 512

def token_count(messages):
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    tokens = tokenizer(text)["input_ids"]
    return len(tokens)


def is_usable(example):
    return (
        example["score_chosen"] > example["score_rejected"]
        and token_count(example["chosen"]) <= MAX_TOKENS
        and token_count(example["rejected"]) <= MAX_TOKENS
    )

### Create the training and validation sets

Filtering a random pool of 6,000 rows is much faster than filtering all 62,135. The
validation set holds 200 pairs that the model never trains on.

In [7]:
pool = train.shuffle(seed=42).select(range(6000))
filtered = pool.filter(is_usable)
usable = filtered.select(range(1700))
usable_data_pairs = usable.train_test_split(test_size=200, seed=42)
train_pairs = usable_data_pairs['train']
eval_pairs = usable_data_pairs['test']

Filter:   0%|          | 0/6000 [00:00<?, ? examples/s]

### Add more training data

This adds 500 new pairs, for 2,000 in total, and keeps the same validation pairs so
results stay comparable.

In [8]:
from datasets import concatenate_datasets

extra = filtered.select(range(1700, 2200))
bigger_train = concatenate_datasets([train_pairs, extra])
print(len(bigger_train))

2000


## 4. Understand the DPO loss

DPO compares how much each answer's log-probability changed from a frozen copy of the
original model:

```text
advantage = (model_chosen - reference_chosen) - (model_rejected - reference_rejected)
loss = -log(sigmoid(beta * advantage))
```

This function is for learning how the loss behaves. TRL's `DPOTrainer` handles the
actual training below.

In [9]:
import torch
import torch.nn.functional as F

def dpo_loss(model_chosen, model_rejected, ref_chosen, ref_rejected, beta=0.1):
    chosen_change = model_chosen - ref_chosen
    rejected_change = model_rejected - ref_rejected

    advantage = chosen_change - rejected_change
    return -F.logsigmoid(beta * advantage)

### Test the loss on three cases

| Case | Advantage | Loss | Meaning |
|---|---|---|---|
| A | 0 | 0.693 | Nothing changed, so this is the starting line |
| B | 10 | 0.313 | The model moved the right way |
| C | -10 | 1.313 | The model moved the wrong way |

In [10]:
ref_c, ref_r = torch.tensor(-50.0), torch.tensor(-60.0)

print(dpo_loss(torch.tensor(-50.0), torch.tensor(-60.0), ref_c, ref_r))  # A: nothing changed
print(dpo_loss(torch.tensor(-45.0), torch.tensor(-65.0), ref_c, ref_r))  # B: good change
print(dpo_loss(torch.tensor(-55.0), torch.tensor(-55.0), ref_c, ref_r))  # C: bad change

tensor(0.6931)
tensor(0.3133)
tensor(1.3133)


## 5. Save a baseline

### Load the model

In [11]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct", dtype=torch.float32
).to("cuda")
print(torch.cuda.get_device_name(0))

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Tesla T4


### Create a helper for generating answers

In [12]:
@torch.inference_mode()
def generate(messages, max_new_tokens=160):
    model.eval()
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=None,
        top_p=None,
        top_k=None,
    )
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

### Record the original answers

These three prompts come from the validation set. `[:-1]` removes the stored answer so
the model sees only the prompt.

In [13]:
pairs = eval_pairs[:3]
sample_prompts = [conversation[:-1] for conversation in pairs["chosen"]]
baseline_answers = [generate(prompt) for prompt in sample_prompts]

In [14]:
for prompt, answer in zip(sample_prompts, baseline_answers):
    print(prompt)
    print(answer)
    print()

[{'content': 'Can you summarize the purpose of the reporting prototypes created by the ESUP Reporting and Data Management (RDM) Team? Answer according to: Recently, staff from the ESUP Reporting and Data Management (RDM) Team created several reporting prototypes and are now sharing them with multiple stakeholder groups to find the best solution that will meet their reporting needs. Sharing the prototypes also provides an opportunity to explore new features available with the system upgrade.\nUsing functional designs from the Student work stream, RDM staff made the prototypes with PeopleSoft (PS) delivered reporting technology to demonstrate what staff can do using real-time data and highlight functionality of the new system.\n“Basically, we want to share the prototypes because we need our stakeholders’ acceptance to use PeopleSoft technology to deliver reports,” said Anderson.\nSo far the response has been positive. “It will be great to have the additional flexibility these reports pro

## 6. Train on 2,000 pairs

This run uses one pass through the data and early stopping, because an earlier run with
a higher learning rate and two passes overfitted. The reference values are computed
once, so a second model doesn't need to stay on the GPU.

The `lm_head.weight` message in the output is expected. Qwen shares those weights with
its embedding layer.

In [15]:
from trl import DPOConfig, DPOTrainer
from transformers import EarlyStoppingCallback

training_args = DPOConfig(
    output_dir="qwen2.5-0.5b-dpo-v3",
    beta=0.1,
    max_length=MAX_TOKENS,
    precompute_ref_log_probs=True,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    learning_rate=2e-6,
    warmup_steps=25,
    fp16=True,
    bf16=False,
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=20,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=1,
    report_to="none",
    save_only_model=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=bigger_train,
    eval_dataset=eval_pairs,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)
trainer.train()
trainer.save_model("qwen2.5-0.5b-dpo-v3")

Extracting prompt from train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Extracting prompt from eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Dropping fully truncated examples from eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

... (12 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 7. Continue on new pairs

Training continues on pairs the model hasn't seen. Passing the original model as
`ref_model` keeps the same reference, so the losses stay comparable. The number printed
before training should match the previous run's best validation loss of about 0.616.

In [16]:
ref_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct", dtype=torch.float32
)
new_pairs = filtered.select(range(2200, len(filtered)))
print(len(new_pairs))

training_args = DPOConfig(
    output_dir="qwen2.5-0.5b-dpo-continued",
    beta=0.1,
    max_length=MAX_TOKENS,
    precompute_ref_log_probs=True,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    learning_rate=2e-6,
    warmup_steps=10,
    fp16=True,
    bf16=False,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=1,
    save_only_model=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=20,
    report_to="none",
)

trainer = DPOTrainer(
    model=model,
    ref_model=ref_model,
    args=training_args,
    train_dataset=new_pairs,
    eval_dataset=eval_pairs,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
print(trainer.evaluate()["eval_loss"])
trainer.train()
trainer.save_model("qwen2.5-0.5b-dpo-continued")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

781


Extracting prompt from train dataset:   0%|          | 0/781 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/781 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/781 [00:00<?, ? examples/s]

Dropping fully truncated examples from eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

... (13 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 8. Review the results

### Validation metrics

Step 50 had the lowest validation loss, at 0.604, so that model was kept.

In [17]:
import pandas as pd

history = pd.DataFrame(
    entry for entry in trainer.state.log_history if "eval_loss" in entry
)
history[["step", "eval_loss", "eval_rewards/accuracies", "eval_rewards/margins"]].round(4)

,step,eval_loss,eval_rewards/accuracies,eval_rewards/margins
0,25,0.6137,0.655,0.5670
1,50,0.6043,0.665,0.5715
2,75,0.6137,0.680,0.5537
3,98,0.6045,0.670,0.6040


### Compare answers before and after training

Each block shows the prompt, the original answer and the final answer.

- **Summary:** longer and uses more of the text, but it invents details.
- **Open question:** more direct and better organized.
- **Arithmetic:** unchanged and still wrong, as the correct answer is `[0, -9]`.

DPO shifted the model toward preferred answers, but that doesn't make every answer
more accurate.

In [18]:
for prompt, answer in zip(sample_prompts, baseline_answers):
    print(prompt)
    print(answer)
    print(generate(prompt))
    print()

[{'content': 'Can you summarize the purpose of the reporting prototypes created by the ESUP Reporting and Data Management (RDM) Team? Answer according to: Recently, staff from the ESUP Reporting and Data Management (RDM) Team created several reporting prototypes and are now sharing them with multiple stakeholder groups to find the best solution that will meet their reporting needs. Sharing the prototypes also provides an opportunity to explore new features available with the system upgrade.\nUsing functional designs from the Student work stream, RDM staff made the prototypes with PeopleSoft (PS) delivered reporting technology to demonstrate what staff can do using real-time data and highlight functionality of the new system.\n“Basically, we want to share the prototypes because we need our stakeholders’ acceptance to use PeopleSoft technology to deliver reports,” said Anderson.\nSo far the response has been positive. “It will be great to have the additional flexibility these reports pro

## 9. Run the final test

The validation pairs helped choose between runs, so this uses 200 new pairs from
outside the original pool.

The final model scored a loss of **0.606** and accuracy of **68.5%**, close to its
validation results. With only 200 pairs, the accuracy could plausibly be 62% to 75%.

In [21]:
test_pairs = (
    train.shuffle(seed=42)
    .select(range(6000, 7000))
    .filter(is_usable)
    .select(range(200))
)

test_results = trainer.evaluate(test_pairs)
print(test_results["eval_loss"], test_results["eval_rewards/accuracies"])

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Extracting prompt from eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Dropping fully truncated examples from eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Computing reference log probs for eval dataset:   0%|          | 0/25 [00:00<?, ?it/s]

Caching reference log probs for eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/200 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/200 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Step,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
0.561694,0.606342,98,1.313516,403849.000000,-2.141351,-2.119408,0.648413,-0.565433,-1.189727,0.685000,0.624294,-221.267477,-208.694100


0.6063424944877625 0.685


## 10. Save the model

Colab deletes local files when the runtime ends, so this copies the model to Google Drive.

In [20]:
from google.colab import drive
import shutil

drive.mount("/content/drive")
shutil.copytree(
    "qwen2.5-0.5b-dpo-continued",
    "/content/drive/MyDrive/qwen2.5-0.5b-dpo-continued",
    dirs_exist_ok=True,
)

Mounted at /content/drive


'/content/drive/MyDrive/qwen2.5-0.5b-dpo-continued'